<a href="https://colab.research.google.com/github/deekay231/freebie/blob/main/kohya_ss_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Train with Kohya's Stable Diffusion Trainers
%cd /content

# Clean up bad parameters, lock stable versions and upgrade Pillow / HuggingFace libraries for Python 3.13
!rm -rf "/content/=4.0.0"
!pip install -U Pillow huggingface-hub diffusers safetensors
!pip install dadaptation==3.1 easygui==0.98.3 einops==0.6.0 fairscale==0.4.13 ftfy==6.1.1 "gradio==3.48.0" "fastapi<0.110.0" "starlette<0.36.0"
!pip install lion-pytorch==0.0.6 lycoris_lora==1.8.0.dev6 open-clip-torch==2.20.0 prodigyopt==1.0 pytorch-lightning==1.9.0 timm==0.6.12
!pip install tk==0.1.0 transformers==4.30.2 voluptuous==0.13.1 wandb==0.15.0 omegaconf

# Install pre-compiled compatible bitsandbytes
!pip install bitsandbytes

%cd /content
!git clone -b v1.0 https://github.com/camenduru/kohya_ss
%cd /content/kohya_ss

!python kohya_gui.py --share --headless

/content
  Using cached pillow-13.0.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-13.0.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.1 MB)
  Attempting uninstall: Pillow
    Found existing installation: pillow 10.4.0
    Uninstalling pillow-10.4.0:
      Successfully uninstalled pillow-10.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 3.48.0 requires pillow<11.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.
  Using cached pillow-10.4.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (9.2 kB)
Using cached pillow-10.4.0-cp313-cp313-manylinux_2_28_x86_64.whl (4.5 MB)
  Attempting uninstall: pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/main.py", line 84, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/__init__.py", line 121, in create_command
    module = importlib.import_module(module_path)
  File "/usr/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "<frozen importlib._bootstrap>", line 1404, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1369, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1340, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 1027, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with

In [8]:
import os

# Apne folders create karein
os.makedirs('/content/dataset/100_maya_woman', exist_ok=True)
os.makedirs('/content/log', exist_ok=True)
os.makedirs('/content/model', exist_ok=True)
print("Folders successfully ban gaye hain!")

Folders successfully ban gaye hain!


In [1]:
from PIL import Image
import os

# Output folder path
output_dir = '/content/dataset/100_maya_woman'
os.makedirs(output_dir, exist_ok=True)

def crop_grid(image_path, rows, cols, prefix):
    if not os.path.exists(image_path):
        print(f"Warning: Image path '{image_path}' nahi mila. Please upload or check path.")
        return
    img = Image.open(image_path)
    width, height = img.size
    cell_w = width // cols
    cell_h = height // rows

    count = 1
    for r in range(rows):
        for c in range(cols):
            box = (c * cell_w, r * cell_h, (c + 1) * cell_w, (r + 1) * cell_h)
            cropped = img.crop(box)
            # Ensure conversion to RGB for JPEGs
            if cropped.mode in ('RGBA', 'LA', 'P'):
                cropped = cropped.convert('RGB')
            cropped.save(f"{output_dir}/{prefix}_{count}.jpg", "JPEG")
            count += 1

# Image 1: 12-Frame Character Sheet (3 rows x 4 cols)
crop_grid('/content/dataset/100_maya_woman/Gemini_Generated_Image_jpuw8xjpuw8xjpuw.jfif', 3, 4, 'frame_12')

# Image 2: 6-Frame Portrait Study (2 rows x 3 cols)
crop_grid('/content/dataset/100_maya_woman/Six-Frame Portrait Study.png', 2, 3, 'frame_6')

print("Sab individual images crop ho kar /content/dataset/100_maya_woman mein save ho gayi hain!")

Sab individual images crop ho kar /content/dataset/100_maya_woman mein save ho gayi hain!


In [2]:
#@title Convert Safetensors to Diffusers
import os

from_safetensors_url = '' #@param {type:"string"}

# Safely download only if URL is provided
if from_safetensors_url.strip():
    print("Downloading safetensors file...")
    !wget {from_safetensors_url} -O /content/model.safetensors
else:
    print("No URL provided. Make sure 'model.safetensors' is uploaded manually to /content/ if you run conversion.")

# Ensure the converter script is available
if not os.path.exists('convert_original_stable_diffusion_to_diffusers.py'):
    !wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/scripts/convert_original_stable_diffusion_to_diffusers.py

# Run the conversion script safely if the model exists
if os.path.exists('/content/model.safetensors'):
    !python3 convert_original_stable_diffusion_to_diffusers.py --half --from_safetensors --checkpoint_path /content/model.safetensors --dump_path /content/model
else:
    print("Error: /content/model.safetensors not found! Please upload your file first or specify a valid URL.")

No URL provided. Make sure 'model.safetensors' is uploaded manually to /content/ if you run conversion.
Error: /content/model.safetensors not found! Please upload your file first or specify a valid URL.


In [ ]:
#@title Push to HF.co

import os
from huggingface_hub import create_repo, upload_folder

hf_token = 'HF_WRITE_TOKEN' #@param {type:"string"}
repo_id = 'username/reponame' #@param {type:"string"}
commit_message = '\u2764' #@param {type:"string"}
create_repo(repo_id, private=True, token=hf_token)
model_path = '/content/train/model' #@param {type:"string"}
upload_folder(folder_path=f'{model_path}', path_in_repo='', repo_id=repo_id, commit_message=commit_message, token=hf_token)

In [ ]:
#@title Push to DagsHub.com

!pip -q install dagshub
from dagshub.upload import Repo, create_repo

repo_id = 'reponame' #@param {type:"string"}
org_name = 'orgname' #@param {type:"string"}
commit_message = '\u2764' #@param {type:"string"}
create_repo(f"{repo_id}", org_name=f"{org_name}")
repo = Repo(f"{org_name}", f"{repo_id}")
model_path = '/content/train/model' #@param {type:"string"}
repo.upload("/content/models", remote_path="data", commit_message=f"{commit_message}", force=True)